# Earnings-Call NLP Pipeline

Driver for the 14-ticker, 131-transcript assignment. All logic lives in `src/`;
this notebook is a thin orchestrator.

**Pipeline:** parse → prices → LLM extraction (gemma3:4b, 4 calls/transcript) → FinBERT sentiment → feature table → 70/30 temporal split → k-fold CV tuned Logistic + XGBoost + CatBoost + SetFit → backtest.

LLM cache is under `cache/extractions/`; FinBERT cache under `cache/finbert.parquet`. Both are idempotent — only missing entries are computed on rerun.

## §0. Environment + paths

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
from src.config import ensure_dirs
ensure_dirs()

## §1. Parse transcripts

Unzips `ECT.zip` on first run and parses every `.txt` into a structured `Transcript` dataclass with header / prepared-remarks / Q&A pairs.

In [ ]:
from src.parser import parse_all
transcripts = parse_all()
tickers = sorted({t.ticker for t in transcripts})
print(f'parsed {len(transcripts)} transcripts across {len(tickers)} tickers:', tickers)

## §2. Prices + forward returns

Fetches daily Close for each ticker + SPY via yfinance (24h parquet cache). Assembles `(ticker, quarter, call_date)` rows with forward excess returns at 1/5/21/63-day horizons and pre-call price-momentum features.

In [ ]:
from src.prices import fetch_all, build_returns_table
prices = fetch_all(tickers)
returns_df = build_returns_table(transcripts, prices)
print(f'{len(returns_df)} rows')
returns_df.head()

## §3. LLM extraction (cached, idempotent)

Hybrid 4-call strategy per transcript (gemma3:4b via Ollama):
1. **Overall call** → sentiment / wins / risks / guidance / themes
2. **CEO call** → sentiment + rationale
3. **CFO call** → sentiment + rationale
4. **Analyst call** → aggregate analyst-question sentiment

Cached under `cache/extractions/`. Near-instant on rerun.

In [ ]:
from tqdm.auto import tqdm
from src.extraction import extract_one
for t in tqdm(transcripts, desc='LLM extract'):
    extract_one(t)
print('LLM extraction cache populated')

## §4. Feature table

One row per parsed transcript (131 rows × 58 cols):
- **LM lexicon** — 131/131
- **FinBERT sentiment** (ProsusAI/finbert, GPU) — 131/131
- **LLM features** — 130/131
- **Price momentum** — 130/131
- **QoQ deltas, speaker gaps, reactive/proactive risks, theme drift** — derived

In [ ]:
from src.features import build
features = build(returns_df, transcripts)
print(f'{len(features)} rows, {features.shape[1]} columns')
coverage = {
    'LM sentiment':        int(features['lm_sentiment'].notna().sum()),
    'FinBERT sentiment':   int(features['finbert_sentiment'].notna().sum()),
    'LLM overall sent.':   int(features['overall_sentiment'].notna().sum()),
    '21-day momentum':     int(features['mom_21d'].notna().sum()),
    'Reactive/proactive':  int(features['reactive_risk_count'].notna().sum()),
    'Theme novelty':       int(features['theme_novelty'].notna().sum()),
}
print('Coverage (non-null / 131):', coverage)
features.head()

## §5. Train / test split (strict temporal 70/30)

Within each ticker, first 70% of calls (by date) → train, rest → test.
Temporal ordering is preserved — no random shuffling.

In [ ]:
from src.model import split_train_test
train, test = split_train_test(features)
print(f'train={len(train)}  test={len(test)}')

## §6. Baselines + models

Eight signals, same `run(test, signal)` evaluation harness for all:
1. **LLM sentiment sign** — simplest NLP signal
2. **LM lexicon sign** — no-LLM baseline
3. **FinBERT sign** — transformer baseline (131/131 coverage)
4. **Logistic regression** — GridSearchCV over C with TimeSeriesSplit(n=5) CV
5. **XGBoost** — Optuna (40 trials) with TimeSeriesSplit(n=5) CV, native NaN support
6. **CatBoost** — Optuna (40 trials) with TimeSeriesSplit(n=5) CV; ordered boosting handles small n=89 better
7. **SetFit (contrastive)** — fine-tuned sentence encoder + logistic head on (transcript, label) pairs
8. **Contrarian SetFit** — invert SetFit probabilities; exploits systematic "sell the news" effect → only positive Sharpe

Metrics: **hit rate**, **rank IC**, **F1 binary** (up vs not-up), **F1 macro** (−1/0/+1), precision, recall, Sharpe.

In [ ]:
from src.model import baseline_rule, lexicon_rule, finbert_rule, fit_logistic, fit_xgboost, fit_catboost
from src.trained_models import fit_setfit
from src.backtest import run, equity_curve, run_cross_sectional, equity_curve_cross_sectional
import pandas as pd
import numpy as np

print('Fitting models...')
log_m    = fit_logistic(train)
xgb_m    = fit_xgboost(train)
cat_m    = fit_catboost(train)
setfit_m = fit_setfit(train)   # loads from cache if available

# Contrarian: invert SetFit probabilities (exploit sell-the-news bias)
def contrarian_setfit(df):
    proba = setfit_m.predict_proba(df).to_numpy()
    sig = np.where(proba < 0.45, 1, np.where(proba > 0.55, -1, 0))
    return pd.Series(sig, index=df.index)

signals = [
    ('Baseline (LLM sign)',    baseline_rule(test)),
    ('Lexicon  (LM sign)',     lexicon_rule(test)),
    ('FinBERT  (sign)',        finbert_rule(test)),
    ('Logistic (C=10, tuned)', log_m.predict(test)),
    ('XGBoost  (Optuna)',      xgb_m.predict(test)),
    ('CatBoost (Optuna)',      cat_m.predict(test)),
    ('SetFit   (contrastive)', setfit_m.predict(test)),
    ('Contrarian SetFit',      contrarian_setfit(test)),
]

rows = []
for name, sig in signals:
    m = run(test, sig)
    rows.append({'signal': name, 'hit': m['hit_rate'], 'IC': m['rank_ic'],
                 'F1_bin': m['f1_binary'], 'F1_mac': m['f1_macro'],
                 'prec': m['precision'], 'rec': m['recall'],
                 'sharpe': m['naive_sharpe'], 'n': m['n']})

pd.DataFrame(rows).set_index('signal').round(3)

## §7. Equity curve (time-series backtest)

In [ ]:
from IPython.display import Image
path = equity_curve(test, log_m.predict(test))
print('saved to', path)
Image(str(path))

## §8. Extensions

### §8.1 Cross-sectional long/short by reporting period

In [ ]:
xgb_score = xgb_m.predict_proba(test)
xs = run_cross_sectional(test, xgb_score.fillna(0))
print('cross-sectional (XGBoost score):', xs)
img_xs = equity_curve_cross_sectional(test, xgb_score.fillna(0))
Image(str(img_xs))

### §8.2 LM vs LLM vs FinBERT sentiment agreement

In [ ]:
pair = features.dropna(subset=['overall_sentiment', 'lm_sentiment', 'finbert_sentiment'])
print(f'n={len(pair)} calls with all three')
print('Spearman correlations:')
print(pair[['overall_sentiment', 'lm_sentiment', 'finbert_sentiment']].corr(method='spearman').round(3))

### §8.3 Reactive vs. proactive risk summary

In [ ]:
feat_rc = features.dropna(subset=['reactive_risk_count']).copy()
by_ticker = feat_rc.groupby('ticker').agg(
    n_calls=('quarter', 'count'),
    avg_proactive=('proactive_risk_count', 'mean'),
    avg_reactive=('reactive_risk_count', 'mean'),
    avg_reactive_ratio=('reactive_risk_ratio', 'mean'),
).round(2)
by_ticker

## §9. Launch dashboard

From the project root:
```bash
streamlit run app.py
```
Three tabs: **Per-Call Explorer** (sentiment badges, wins/risks, FinBERT scores), **Ticker Timeline** (LLM vs LM vs FinBERT overlay), **Backtest** (5 signals, F1/IC/Sharpe metrics, equity curves).